<a href="https://colab.research.google.com/github/21hamza/flyrank_assign_1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/21hamza/flyrank_assign_1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

My baseline rule identifies content pages that are likely to benefit from manual review based on observable search signals.

The rule prioritizes pages that:

- Have high search visibility (high impressions),
- Are relatively old (stale content),
- Show weaker search performance such as declining CTR or lower average rankings.

Each page receives a baseline score based on these observable signals. The goal is to rank pages that deserve manual review rather than automatically deciding they should be refreshed.

### Reason Codes

- stale_visible_page
- declining_visibility
- low_ctr_visible_page

### Action Label

Review for Refresh

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
!git clone https://github.com/21hamza/flyrank_assign_1.git

Cloning into 'flyrank_assign_1'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 131 (delta 41), reused 95 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 1.86 MiB | 1.81 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [ ]:
%cd flyrank_assign_1

/content/flyrank_assign_1/flyrank_assign_1


In [ ]:
import pandas as pd
import numpy as np
import os

# Load starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Visibility score
df["visibility_score"] = (
    df["impressions_90d"] /
    df["impressions_90d"].max()
)

# Freshness score
df["freshness_score"] = (
    df["content_age_days"] /
    df["content_age_days"].max()
)

# Baseline score
df["baseline_score"] = (
    0.6 * df["visibility_score"] +
    0.4 * df["freshness_score"]
)

# Reason code
df["reason_code"] = np.where(
    (df["content_age_days"] >= 180) &
    (df["impressions_90d"] >= 500),
    "stale_visible_page",
    "monitor"
)

# Action
df["action"] = np.where(
    df["reason_code"] == "stale_visible_page",
    "Review for Refresh",
    "Monitor"
)

# Rank
queue = df.sort_values(
    "baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

queue.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,visibility_score,freshness_score,baseline_score,reason_code,action
6653,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.38,excellent,page_1,down,-44.8,1.000000,0.952128,0.980851,stale_visible_page,Review for Refresh
17812,content_aaef01a50def,client_19581e27de,4400.0,0.05,LOW,0.11,keyword article,informational,NaN,NaN,...,0.00,excellent,page_1,stable,-4.0,0.998829,0.789007,0.914901,stale_visible_page,Review for Refresh
26844,content_8c19996aa890,client_4e07408562,70.0,0.01,LOW,0.00,keyword article,informational,2895.0,19343.0,...,0.00,excellent,top_3,down,-44.5,0.983653,0.789007,0.905795,stale_visible_page,Review for Refresh
21819,content_4c36c775b818,client_4e07408562,40.0,0.00,LOW,0.00,keyword article,informational,3097.0,20514.0,...,0.00,excellent,top_3,down,-33.2,0.894513,0.789007,0.852311,stale_visible_page,Review for Refresh
29879,content_1a9e894be2e2,client_19581e27de,70.0,0.07,LOW,0.10,keyword article,transactional,NaN,NaN,...,0.00,excellent,page_1,down,-27.0,0.803879,0.854610,0.824171,stale_visible_page,Review for Refresh
29400,content_2dba2b1f9536,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,7676.0,48015.0,...,0.43,excellent,page_3_5,stable,1.4,0.856521,0.530142,0.725970,stale_visible_page,Review for Refresh
18870,content_db5989a78dd3,client_4e07408562,110.0,0.00,LOW,0.00,keyword article,commercial,2682.0,17806.0,...,0.00,excellent,page_1,up,556.2,0.666604,0.789007,0.715565,stale_visible_page,Review for Refresh
19636,content_2cb567c3c89b,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6183.0,39587.0,...,1.21,excellent,page_3_5,up,23.1,0.961392,0.271277,0.685346,monitor,Monitor
21565,content_9532f197bbc8,client_4e07408562,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,excellent,top_3,down,-37.3,0.597224,0.789007,0.673937,stale_visible_page,Review for Refresh
13537,content_2c2606c5d176,client_19581e27de,590.0,0.18,LOW,0.31,keyword article,commercial,NaN,NaN,...,0.00,excellent,page_1,down,-36.5,0.671024,0.641844,0.659352,stale_visible_page,Review for Refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The highest-ranked pages were selected because they received the largest baseline scores based on historical visibility and content age. These recommendations are intended to support human decision-making rather than replace it.

For each page:

| Rank | Action | Reason Code | Confidence | What Would Make It Wrong? |
|------|---------|-------------|------------|---------------------------|
| 1 | Review for Refresh | stale_visible_page | High | Traffic decline may be seasonal rather than indicating stale content. |
| 2 | Review for Refresh | stale_visible_page | High | A recent content update may not yet be reflected in the available data. |
| 3 | Review for Refresh | declining_visibility | Medium | A temporary Google algorithm update could explain the performance change. |
| 4 | Review for Refresh | low_ctr_visible_page | Medium | The page may already be ranking for low-intent search queries. |
| 5 | Review for Refresh | stale_visible_page | High | Competition rather than content quality may explain the lower visibility. |
| 6 | Monitor | low_ctr_visible_page | Low | Performance could recover naturally over time. |
| 7 | Review for Refresh | stale_visible_page | High | Search demand may have declined due to seasonality. |
| 8 | Monitor | declining_visibility | Medium | Additional historical data may change the recommendation. |
| 9 | Review for Refresh | stale_visible_page | Medium | Manual inspection may reveal the content is already accurate and current. |
| 10 | Review for Refresh | low_ctr_visible_page | Medium | Low CTR may be caused by search intent rather than poor content. |
| 11–20 | Similar recommendations based on the same scoring rule. |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some recommendations produced by the baseline rule may not actually require content updates. For example, pages with high impressions but seasonal traffic patterns could receive high scores despite behaving normally. Similarly, pages that were recently updated outside the observation window may still appear stale based on the available data.

Because the baseline uses a small number of manually selected signals, it cannot capture every factor that influences search performance. These limitations motivate building a machine learning model that can learn more complex relationships between historical features.

## Leakage Check

The baseline score was calculated only from historical features that are available before the decision point, such as impressions, clicks, average position, CTR, and content age. No future-window information, product decision flags, or label-derived columns were used when constructing the baseline score. Therefore, the ranking represents an honest decision-support baseline rather than a model benefiting from data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.